In [ ]:
"""
AlphaGPT + LightGBM 混合量化策略生成系统（增强版）
一键运行：数据加载 → 特征工程 → AlphaGPT训练 → 策略集成 → 可视化报告
新增功能：
- 更多技术因子（MACD, Bollinger, ATR, RSI, OBV等）
- LightGBM超参数调优（使用Optuna）
- 风控模块（止损、波动率调整仓位、最大持仓时间）
- 改进的Plotly图表（可保存为HTML/PNG）
"""

import os
import math
import warnings
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split, TimeSeriesSplit
import optuna  # 用于超参数优化

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import tushare as ts

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.sans-serif'] = ['SimHei']


# ==================== 全局配置 ====================
TS_TOKEN = '0eaedd58fde7cd9bb30751d00e3216de97b287b6e1adb55d1fd06edc'
INDEX_CODE = '000905.SH'
START_DATE = '20150101'
END_DATE = '20250101'
TEST_END_DATE = '20260225'

BATCH_SIZE = 512
TRAIN_ITERATIONS = 400
MAX_SEQ_LEN = 8
COST_RATE = 0.0005
SLIPPAGE = 0.0002
DATA_CACHE_PATH = f'/home/gash000/msc/quant-project/src/python/AlphaGPT-main/data_cache_{INDEX_CODE}.parquet'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')

# 风控参数
STOP_LOSS = 0.05          # 单日最大亏损阈值（超过则强制平仓）
MAX_POSITION_DAYS = 10    # 最大连续持仓天数（强制换仓）
VOLATILITY_TARGET = 0.20  # 目标年化波动率（用于仓位调整）

# ==================== 核心算子（技术指标） ====================
@torch.jit.script
def _ts_delay(x: torch.Tensor, d: int) -> torch.Tensor:
    if d == 0: return x
    pad = torch.zeros((x.shape[0], d), device=x.device)
    return torch.cat([pad, x[:, :-d]], dim=1)

@torch.jit.script
def _ts_delta(x: torch.Tensor, d: int) -> torch.Tensor:
    return x - _ts_delay(x, d)

@torch.jit.script
def _ts_zscore(x: torch.Tensor, d: int) -> torch.Tensor:
    if d <= 1: return torch.zeros_like(x)
    B, T = x.shape
    pad = torch.zeros((B, d - 1), device=x.device)
    x_pad = torch.cat([pad, x], dim=1)
    windows = x_pad.unfold(1, d, 1)
    mean = windows.mean(dim=-1)
    std = windows.std(dim=-1) + 1e-6
    return (x - mean) / std

@torch.jit.script
def _ts_decay_linear(x: torch.Tensor, d: int) -> torch.Tensor:
    if d <= 1: return x
    B, T = x.shape
    pad = torch.zeros((B, d - 1), device=x.device)
    x_pad = torch.cat([pad, x], dim=1)
    windows = x_pad.unfold(1, d, 1)
    w = torch.arange(1, d + 1, device=x.device, dtype=x.dtype)
    w = w / w.sum()
    return (windows * w).sum(dim=-1)

@torch.jit.script
def _ts_correlation(x: torch.Tensor, y: torch.Tensor, d: int) -> torch.Tensor:
    if d <= 1: return torch.zeros_like(x)
    B, T = x.shape
    pad = torch.zeros((B, d - 1), device=x.device)
    x_pad = torch.cat([pad, x], dim=1)
    y_pad = torch.cat([pad, y], dim=1)
    x_windows = x_pad.unfold(1, d, 1)
    y_windows = y_pad.unfold(1, d, 1)
    x_mean = x_windows.mean(dim=-1, keepdim=True)
    y_mean = y_windows.mean(dim=-1, keepdim=True)
    cov = ((x_windows - x_mean) * (y_windows - y_mean)).mean(dim=-1)
    x_std = x_windows.std(dim=-1) + 1e-6
    y_std = y_windows.std(dim=-1) + 1e-6
    return cov / (x_std * y_std)

@torch.jit.script
def _ts_rank(x: torch.Tensor, d: int) -> torch.Tensor:
    if d <= 1: return torch.zeros_like(x)
    B, T = x.shape
    pad = torch.zeros((B, d - 1), device=x.device)
    x_pad = torch.cat([pad, x], dim=1)
    windows = x_pad.unfold(1, d, 1)
    ranks = torch.zeros_like(windows)
    for b in range(B):
        for t in range(T):
            window_data = windows[b, t]
            sorted_indices = torch.argsort(window_data)
            rank = torch.zeros_like(window_data)
            rank[sorted_indices] = torch.arange(len(window_data), device=x.device, dtype=x.dtype)
            ranks[b, t] = rank
    return ranks[:, :, -1] / (d - 1)

# ==================== 运算符配置（完整扩展版） ====================
OPS_CONFIG = [
    ('ADD', lambda x, y: x + y, 2),
    ('SUB', lambda x, y: x - y, 2),
    ('MUL', lambda x, y: x * y, 2),
    ('DIV', lambda x, y: x / (y + 1e-6 * torch.sign(y)), 2),
    ('MAX', lambda x, y: torch.maximum(x, y), 2),
    ('MIN', lambda x, y: torch.minimum(x, y), 2),
    ('NEG', lambda x: -x, 1),
    ('ABS', lambda x: torch.abs(x), 1),
    ('SIGN', lambda x: torch.sign(x), 1),
    ('SQRT', lambda x: torch.sqrt(torch.abs(x) + 1e-6), 1),
    ('SQUARE', lambda x: x ** 2, 1),
    ('LOG', lambda x: torch.log(torch.abs(x) + 1), 1),
    ('DELTA5', lambda x: _ts_delta(x, 5), 1),
    ('DELTA10', lambda x: _ts_delta(x, 10), 1),
    ('MA5', lambda x: _ts_decay_linear(x, 5), 1),
    ('MA10', lambda x: _ts_decay_linear(x, 10), 1),
    ('MA20', lambda x: _ts_decay_linear(x, 20), 1),
    ('STD20', lambda x: _ts_zscore(x, 20), 1),
    ('RANK20', lambda x: _ts_rank(x, 20), 1),
    ('CORR10', lambda x, y: _ts_correlation(x, y, 10), 2),
    ('CORR20', lambda x, y: _ts_correlation(x, y, 20), 2),
]

# 基础特征（动态添加LGB特征）
BASE_FEATURES = [
    'RET', 'RET5', 'RET10', 'RET20',
    'VOL_CHG', 'VOL_RET', 'TREND',
    'HIGH_LOW', 'RSI14', 'VOLATILITY',
    # 新增因子
    'MACD', 'BB_WIDTH', 'ATR', 'OBV', 'CCI', 'ADX'
]

# ==================== 增强数据引擎（含LightGBM特征工程） ====================
class EnhancedDataEngine:
    FEATURES = BASE_FEATURES.copy()          # 类变量，可动态更新
    OPS = OPS_CONFIG

    def __init__(self):
        self.pro = ts.pro_api(TS_TOKEN)
        self.dates = None
        self.feat_data = None
        self.target_oto_ret = None
        self.raw_open = None
        self.raw_close = None
        self.raw_high = None
        self.raw_low = None
        self.raw_vol = None
        self.split_idx = None
        self.lgb_model = None
        self.important_features_idx = None

    def calculate_rsi(self, prices, period=14):
        delta = pd.Series(prices).diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-6)
        rsi = 100 - (100 / (1 + rs))
        return np.nan_to_num(rsi.values).astype(np.float32)

    def calculate_macd(self, prices, fast=12, slow=26, signal=9):
        ema_fast = pd.Series(prices).ewm(span=fast, adjust=False).mean()
        ema_slow = pd.Series(prices).ewm(span=slow, adjust=False).mean()
        macd_line = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        macd = macd_line - signal_line
        return np.nan_to_num(macd.values).astype(np.float32)

    def calculate_bollinger_width(self, prices, period=20, num_std=2):
        sma = pd.Series(prices).rolling(period).mean()
        std = pd.Series(prices).rolling(period).std()
        upper = sma + num_std * std
        lower = sma - num_std * std
        width = (upper - lower) / (sma + 1e-6)
        return np.nan_to_num(width.values).astype(np.float32)

    def calculate_atr(self, high, low, close, period=14):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
        tr1 = high - low
        tr2 = (high - close.shift()).abs()
        tr3 = (low - close.shift()).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.rolling(period).mean()
        return np.nan_to_num(atr.values).astype(np.float32)

    def calculate_obv(self, close, volume):
        obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
        return np.nan_to_num(obv.values).astype(np.float32)

    def calculate_cci(self, high, low, close, period=20):
        tp = (high + low + close) / 3
        sma = tp.rolling(period).mean()
        mad = tp.rolling(period).apply(lambda x: np.abs(x - x.mean()).mean())
        cci = (tp - sma) / (0.015 * mad + 1e-6)
        return np.nan_to_num(cci.values).astype(np.float32)

    def calculate_adx(self, high, low, close, period=14):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
        plus_dm = high.diff()
        minus_dm = low.diff()
        plus_dm[plus_dm < 0] = 0
        minus_dm[minus_dm > 0] = 0
        tr = pd.concat([high - low, (high - close.shift()).abs(), (low - close.shift()).abs()], axis=1).max(axis=1)
        atr = tr.rolling(period).mean()
        plus_di = 100 * (plus_dm.rolling(period).mean() / (atr + 1e-6))
        minus_di = 100 * (-minus_dm.rolling(period).mean() / (atr + 1e-6))
        dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-6))
        adx = dx.rolling(period).mean()
        return np.nan_to_num(adx.values).astype(np.float32)

    def robust_norm(self, x):
        x = x.astype(np.float32)
        median = np.nanmedian(x)
        mad = np.nanmedian(np.abs(x - median)) + 1e-6
        res = (x - median) / mad
        return np.clip(res, -5, 5).astype(np.float32)

    def load(self, use_lgb=True):
        """加载数据，并可选进行LightGBM特征增强"""
        # ----- 基础数据加载 -----
        if os.path.exists(DATA_CACHE_PATH):
            df = pd.read_parquet(DATA_CACHE_PATH)
        else:
            print(f"🌐 从Tushare获取 {INDEX_CODE} 数据...")
            if INDEX_CODE.endswith('BTC'):
                df = pd.read_csv('/home/gash000/downloads/btc_1d.csv')
                df.columns = ['trade_date', 'open', 'high', 'low', 'close', 'volume']
                df['trade_date'] = df['trade_date'].apply(lambda x: str(x))
            else:
                df = self.pro.index_daily(ts_code=INDEX_CODE, start_date=START_DATE, end_date=TEST_END_DATE)
            if df is None or df.empty:
                raise ValueError("数据获取失败，请检查Token和网络")
            df = df.sort_values('trade_date').reset_index(drop=True)
            df.to_parquet(DATA_CACHE_PATH)

        for col in ['open', 'high', 'low', 'close', 'vol']:
            df[col] = pd.to_numeric(df[col], errors='coerce').ffill().bfill()

        self.dates = pd.to_datetime(df['trade_date'])
        close = df['close'].values.astype(np.float32)
        open_ = df['open'].values.astype(np.float32)
        high = df['high'].values.astype(np.float32)
        low = df['low'].values.astype(np.float32)
        vol = df['vol'].values.astype(np.float32)

        # 保存原始数据用于风控
        self.raw_open = torch.from_numpy(open_).to(DEVICE)
        self.raw_close = torch.from_numpy(close).to(DEVICE)
        self.raw_high = torch.from_numpy(high).to(DEVICE)
        self.raw_low = torch.from_numpy(low).to(DEVICE)
        self.raw_vol = torch.from_numpy(vol).to(DEVICE)

        # ----- 基础特征计算 -----
        ret = np.zeros_like(close); ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-6)
        ret5 = pd.Series(close).pct_change(5).fillna(0).values.astype(np.float32)
        ret10 = pd.Series(close).pct_change(10).fillna(0).values.astype(np.float32)
        ret20 = pd.Series(close).pct_change(20).fillna(0).values.astype(np.float32)

        vol_ma = pd.Series(vol).rolling(20).mean().values
        vol_chg = np.zeros_like(vol)
        mask = vol_ma > 0
        vol_chg[mask] = vol[mask] / vol_ma[mask] - 1
        vol_chg = np.nan_to_num(vol_chg).astype(np.float32)
        v_ret = (ret * (vol_chg + 1)).astype(np.float32)

        ma60 = pd.Series(close).rolling(60).mean().values
        trend = np.zeros_like(close)
        mask = ma60 > 0
        trend[mask] = close[mask] / ma60[mask] - 1
        trend = np.nan_to_num(trend).astype(np.float32)

        high_low = (high - low) / (close + 1e-6)
        high_low = np.nan_to_num(high_low).astype(np.float32)

        rsi14 = self.calculate_rsi(close, 14)
        volatility = pd.Series(close).pct_change().rolling(20).std().fillna(0).values.astype(np.float32)

        # 新增因子
        macd = self.calculate_macd(close)
        bb_width = self.calculate_bollinger_width(close)
        atr = self.calculate_atr(high, low, close)
        obv = self.calculate_obv(pd.Series(close), pd.Series(vol))
        cci = self.calculate_cci(pd.Series(high), pd.Series(low), pd.Series(close))
        adx = self.calculate_adx(pd.Series(high), pd.Series(low), pd.Series(close))

        # ----- 归一化并构建初始特征张量 -----
        feature_list = [
            self.robust_norm(ret),
            self.robust_norm(ret5),
            self.robust_norm(ret10),
            self.robust_norm(ret20),
            self.robust_norm(vol_chg),
            self.robust_norm(v_ret),
            self.robust_norm(trend),
            self.robust_norm(high_low),
            self.robust_norm(rsi14),
            self.robust_norm(volatility),
            self.robust_norm(macd),
            self.robust_norm(bb_width),
            self.robust_norm(atr),
            self.robust_norm(obv),
            self.robust_norm(cci),
            self.robust_norm(adx)
        ]
        self.feat_data = torch.stack([torch.from_numpy(f).to(DEVICE) for f in feature_list])

        # ----- 目标收益率（Open-to-Open）-----
        open_tensor = self.raw_open
        open_t1 = torch.roll(open_tensor, -1)
        open_t2 = torch.roll(open_tensor, -2)
        self.target_oto_ret = (open_t2 - open_t1) / (open_t1 + 1e-6)
        self.target_oto_ret[-2:] = 0.0

        # 训练/测试分割（80/20）
        self.split_idx = int(len(df) * 0.8)
        print(f"✅ 基础数据加载完成，样本数：{len(df)}，训练集：0-{self.split_idx}，测试集：{self.split_idx}-{len(df)}")

        # ----- LightGBM特征增强（可选）-----
        if use_lgb:
            self._add_lgb_features()

        return self

    def _add_lgb_features(self):
        """使用LightGBM生成组合特征并添加到特征张量"""
        print("🔍 训练LightGBM进行特征选择与组合...")
        split = self.split_idx
        X_train = self.feat_data[:, :split].T.cpu().numpy()
        y_train = self.target_oto_ret[:split].cpu().numpy()

        # 使用Optuna进行超参数优化
        def objective(trial):
            params = {
                'boosting_type': 'gbdt',
                'objective': 'regression',
                'metric': 'rmse',
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
                'verbose': -1,
                'num_threads': 4,
                'seed': 42
            }
            cv = TimeSeriesSplit(n_splits=3)
            scores = []
            for train_idx, val_idx in cv.split(X_train):
                X_tr, X_val = X_train[train_idx], X_train[val_idx]
                y_tr, y_val = y_train[train_idx], y_train[val_idx]
                lgb_train = lgb.Dataset(X_tr, y_tr)
                lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
                model = lgb.train(params, lgb_train, valid_sets=[lgb_val], num_boost_round=100,
                                  callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)])
                scores.append(model.best_score['valid_0']['rmse'])
            return np.mean(scores)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=20, show_progress_bar=True)
        best_params = study.best_params
        best_params.update({
            'boosting_type': 'gbdt',
            'objective': 'regression',
            'metric': 'rmse',
            'verbose': -1,
            'num_threads': 4,
            'seed': 42
        })
        print(f"   最佳LightGBM参数: {best_params}")

        lgb_train = lgb.Dataset(X_train, y_train)
        self.lgb_model = lgb.train(
            best_params,
            lgb_train,
            num_boost_round=200,
            valid_sets=[lgb_train],
            callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
        )

        # 特征重要性筛选（取前50%重要特征）
        feature_importance = self.lgb_model.feature_importance(importance_type='gain')
        threshold = np.percentile(feature_importance, 50)
        self.important_features_idx = [i for i, imp in enumerate(feature_importance) if imp > threshold]
        print(f"   筛选出 {len(self.important_features_idx)} 个重要特征")

        # 生成LightGBM预测值作为新特征
        all_preds = self.lgb_model.predict(self.feat_data.T.cpu().numpy())
        lgb_feature = torch.from_numpy(self.robust_norm(all_preds)).float().to(DEVICE).unsqueeze(0)

        # 更新特征张量和特征列表
        self.feat_data = torch.cat([self.feat_data, lgb_feature], dim=0)
        self.__class__.FEATURES = BASE_FEATURES + ['LGB_FEATURE']
        print(f"✅ 添加LightGBM特征，当前特征总数：{len(self.__class__.FEATURES)}")

# ==================== AlphaGPT模型 ====================
class AlphaGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_head=4, n_layer=2, dropout=0.1, max_seq_len=MAX_SEQ_LEN):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len + 1, d_model))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=n_layer)

        self.ln_f = nn.LayerNorm(d_model)
        self.head_actor = nn.Linear(d_model, vocab_size)
        self.head_critic = nn.Linear(d_model, 1)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0, std=0.02)

    def forward(self, idx):
        B, T = idx.size()
        x = self.token_emb(idx) + self.pos_emb[:, :T, :]
        x = self.dropout(x)
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)
        x = self.blocks(x, mask=mask, is_causal=True)
        x = self.ln_f(x)
        last = x[:, -1, :]
        return self.head_actor(last), self.head_critic(last)

# ==================== 深度挖掘器（强化学习训练 + 奖励预测器） ====================
class DeepQuantMiner:
    def __init__(self, engine, vocab_size, op_func_map, op_arity_map, features_list):
        self.engine = engine
        self.vocab_size = vocab_size
        self.op_func_map = op_func_map
        self.op_arity_map = op_arity_map
        self.features_list = features_list

        self.model = AlphaGPT(vocab_size).to(DEVICE)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=3e-4, weight_decay=1e-5, betas=(0.9,0.95))
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=TRAIN_ITERATIONS)

        self.best_sharpe = -10.0
        self.best_formula_tokens = None
        self.patience_counter = 0
        self.max_patience = 20

        # 奖励预测器（LightGBM）
        self.reward_predictor = None

    # ---------- 公式解析 ----------
    def solve_one(self, tokens):
        stack = []
        try:
            for t in reversed(tokens):
                if t < len(self.features_list):
                    stack.append(self.engine.feat_data[t])
                else:
                    arity = self.op_arity_map[t]
                    if len(stack) < arity:
                        raise ValueError
                    args = [stack.pop() for _ in range(arity)]
                    func = self.op_func_map[t]
                    res = func(*args) if arity == 2 else func(args[0])
                    if torch.isnan(res).any() or torch.isinf(res).any():
                        res = torch.nan_to_num(res)
                    stack.append(res)
            if len(stack) >= 1:
                final = stack[-1]
                if final.std() < 1e-4:
                    return None
                return final
        except:
            return None
        return None

    def solve_batch(self, token_seqs):
        B, T = token_seqs.shape[0], self.engine.feat_data.shape[1]
        results = torch.zeros((B, T), device=DEVICE)
        valid_mask = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        for i in range(B):
            res = self.solve_one(token_seqs[i].cpu().tolist())
            if res is not None:
                results[i] = res
                valid_mask[i] = True
        return results, valid_mask

    # ---------- 回测奖励（增强风控版）----------
    def backtest_with_risk_control(self, factors, is_train=True):
        """包含止损、仓位控制的回测"""
        if factors.shape[0] == 0:
            return torch.tensor([], device=DEVICE)
        split = self.engine.split_idx
        target = self.engine.target_oto_ret[:split] if is_train else self.engine.target_oto_ret[split:]
        factor_data = factors[:, :split] if is_train else factors[:, split:]

        # 获取历史波动率（用于仓位调整）
        close_prices = self.engine.raw_close[:split] if is_train else self.engine.raw_close[split:]
        returns = (close_prices[1:] - close_prices[:-1]) / (close_prices[:-1] + 1e-6)

        # 修正波动率计算
        vol_vals = pd.Series(returns.cpu().numpy()).rolling(20).std().fillna(0).values
        hist_vol = torch.zeros_like(close_prices)
        hist_vol[20:] = torch.tensor(vol_vals[19:], device=DEVICE) * np.sqrt(252)

        rewards = torch.zeros(factors.shape[0], device=DEVICE)
        for i in range(factors.shape[0]):
            f = factor_data[i]
            if torch.isnan(f).all() or (f == 0).all() or f.numel() < 20:
                rewards[i] = -3.0
                continue

            # 基础信号
            sig = torch.tanh(f * 0.5)
            # 波动率调整仓位：目标波动率 / 历史波动率（限制在0.2~2之间）
            vol_scale = VOLATILITY_TARGET / (hist_vol + 1e-6)
            vol_scale = torch.clamp(vol_scale, 0.2, 2.0)
            position = sig * vol_scale

            # 强制平仓逻辑：止损（当日亏损超过阈值）或持仓超时
            daily_pnl = position * target
            stop_loss_trigger = (daily_pnl < -STOP_LOSS).float()
            position = position * (1 - stop_loss_trigger)  # 触发止损后当日仓位清零（假设盘后执行）

            # 持仓天数限制（简化：若连续持仓超过MAX_POSITION_DAYS，强制平仓）
            pos_sign = torch.sign(position)
            pos_days = torch.zeros_like(pos_sign)
            for t in range(1, len(pos_sign)):
                if pos_sign[t] == pos_sign[t-1] and pos_sign[t] != 0:
                    pos_days[t] = pos_days[t-1] + 1
                else:
                    pos_days[t] = 0
            force_close = (pos_days > MAX_POSITION_DAYS).float()
            position = position * (1 - force_close)

            # 计算换手率（考虑仓位变化）
            turnover = torch.abs(position - torch.roll(position, 1))
            turnover[0] = 0.0

            pnl = position * target - turnover * (COST_RATE + SLIPPAGE)
            if pnl.numel() < 20:
                rewards[i] = -3.0
                continue

            mu = pnl.mean() * 252
            std = pnl.std() * np.sqrt(252) + 1e-6
            downside = pnl[pnl < 0]
            if downside.numel() > 10:
                down_std = downside.std() * np.sqrt(252) + 1e-6
                sortino = (mu - 0.02) / down_std
            else:
                sortino = (mu - 0.02) / std

            # 惩罚过高换手率
            avg_turnover = turnover.mean()
            if avg_turnover > 0.8:
                sortino -= 2.0
            elif avg_turnover > 0.5:
                sortino -= 1.0
            if (position == 0).all():
                sortino = -3.0

            rewards[i] = torch.clamp(sortino, -5, 10)
        return rewards

    # ---------- 随机公式生成 ----------
    def generate_random_formula(self):
        tokens = []
        open_slots = 1
        for _ in range(MAX_SEQ_LEN):
            if open_slots == 0:
                tokens.append(np.random.randint(0, len(self.features_list)))
            else:
                if np.random.random() < 0.3 and open_slots > 1:
                    op_idx = np.random.randint(len(self.features_list), self.vocab_size)
                    tokens.append(op_idx)
                    open_slots += self.op_arity_map[op_idx] - 1
                else:
                    tokens.append(np.random.randint(0, len(self.features_list)))
                    open_slots -= 1
        return tokens

    # ---------- 公式特征提取（用于奖励预测）----------
    def extract_formula_features(self, factor):
        if factor is None or factor.numel() == 0:
            return np.zeros(5)
        feat = []
        feat.append(factor.mean().item())
        feat.append(factor.std().item())
        factor_np = factor.cpu().numpy()
        feat.append(pd.Series(factor_np).skew())
        feat.append(pd.Series(factor_np).kurtosis())
        if len(factor_np) > 5:
            autocorr = pd.Series(factor_np).autocorr()
            feat.append(autocorr if not np.isnan(autocorr) else 0)
        else:
            feat.append(0)
        return np.array(feat, dtype=np.float32)

    # ---------- 训练LightGBM奖励预测器 ----------
    def train_reward_predictor(self, n_samples=3000):
        print("🎯 训练LightGBM奖励预测器...")
        X, y = [], []
        pbar = tqdm(range(n_samples), desc="生成样本")
        for _ in pbar:
            tokens = self.generate_random_formula()
            factor = self.solve_one(tokens)
            if factor is not None:
                features = self.extract_formula_features(factor[:self.engine.split_idx])
                reward = self.backtest_with_risk_control(factor.unsqueeze(0), is_train=True)[0].item()
                if reward > -2:   # 只保留有一定意义的样本
                    X.append(features)
                    y.append(reward)
        X = np.array(X)
        y = np.array(y)

        if len(X) < 100:
            print("⚠️  样本不足，跳过奖励预测器训练")
            return

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        lgb_train = lgb.Dataset(X_train, y_train)
        lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'num_leaves': 31,
            'learning_rate': 0.01,
            'verbosity': -1,
            'seed': 42
        }
        self.reward_predictor = lgb.train(
            params,
            lgb_train,
            num_boost_round=300,
            valid_sets=[lgb_val],
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
        )
        print(f"✅ 奖励预测器训练完成，验证集RMSE：{self.reward_predictor.best_score['valid_0']['rmse']:.4f}")

    # ---------- 策略生成训练（强化学习）----------
    def get_strict_mask(self, open_slots, step):
        B = open_slots.shape[0]
        mask = torch.full((B, self.vocab_size), float('-inf'), device=DEVICE)
        remaining = MAX_SEQ_LEN - step
        done_mask = (open_slots == 0)
        mask[done_mask, 0] = 0.0
        active = ~done_mask
        must_pick_feat = (open_slots >= remaining)
        mask[active, :len(self.features_list)] = 0.0
        can_pick_op = active & (~must_pick_feat)
        if can_pick_op.any():
            mask[can_pick_op, len(self.features_list):] = 0.0
        return mask

    def train(self, use_reward_predictor=True):
        if use_reward_predictor and self.reward_predictor is None:
            self.train_reward_predictor()

        print(f"🚀 开始AlphaGPT强化学习训练，最大公式长度={MAX_SEQ_LEN}...")
        pbar = tqdm(range(TRAIN_ITERATIONS))

        for iteration in pbar:
            B = BATCH_SIZE
            open_slots = torch.ones(B, dtype=torch.long, device=DEVICE)
            log_probs, tokens, values = [], [], []
            curr_inp = torch.zeros((B, 1), dtype=torch.long, device=DEVICE)

            for step in range(MAX_SEQ_LEN):
                logits, val = self.model(curr_inp)
                mask = self.get_strict_mask(open_slots, step)
                dist = Categorical(logits=(logits + mask))
                action = dist.sample()
                log_probs.append(dist.log_prob(action))
                tokens.append(action)
                values.append(val.squeeze())
                curr_inp = torch.cat([curr_inp, action.unsqueeze(1)], dim=1)

                is_op = action >= len(self.features_list)
                delta = torch.full((B,), -1, device=DEVICE)
                if is_op.any():
                    arity_tens = torch.zeros(self.vocab_size, dtype=torch.long, device=DEVICE)
                    for k, v in self.op_arity_map.items():
                        arity_tens[k] = v
                    op_delta = arity_tens[action] - 1
                    delta = torch.where(is_op, op_delta, delta)
                delta[open_slots == 0] = 0
                open_slots += delta

            seqs = torch.stack(tokens, dim=1)

            # 评估
            with torch.no_grad():
                f_vals, valid_mask = self.solve_batch(seqs)
                valid_idx = torch.where(valid_mask)[0]
                rewards = torch.full((B,), -2.0, device=DEVICE)

                if len(valid_idx) > 0:
                    # 如果奖励预测器存在，先用它快速筛选
                    if self.reward_predictor is not None:
                        pred_rewards = []
                        for idx in valid_idx:
                            factor = f_vals[idx]
                            feats = self.extract_formula_features(factor[:self.engine.split_idx])
                            pred = self.reward_predictor.predict([feats])[0]
                            pred_rewards.append(pred)
                        pred_rewards = torch.tensor(pred_rewards, device=DEVICE)
                        # 只对top-20%做精确回测
                        top_k = max(1, len(pred_rewards) // 5)
                        _, top_inds = torch.topk(pred_rewards, top_k)
                        candidate_idx = valid_idx[top_inds.cpu()]
                    else:
                        candidate_idx = valid_idx

                    bt_scores = self.backtest_with_risk_control(f_vals[candidate_idx], is_train=True)
                    rewards[candidate_idx] = bt_scores

                    # 更新最佳策略
                    best_sub_idx = torch.argmax(bt_scores)
                    current_best = bt_scores[best_sub_idx].item()
                    if current_best > self.best_sharpe:
                        self.best_sharpe = current_best
                        self.best_formula_tokens = seqs[candidate_idx[best_sub_idx]].cpu().tolist()
                        self.patience_counter = 0
                        # 测试集验证
                        test_factor = self.solve_one(self.best_formula_tokens)
                        if test_factor is not None:
                            test_reward = self.backtest_with_risk_control(test_factor.unsqueeze(0), is_train=False)[0].item()
                            print(f"\n🎯 新最佳策略：训练奖励={current_best:.3f}，测试奖励={test_reward:.3f}")
                    else:
                        self.patience_counter += 1

            # 策略优化（PPO风格）
            returns = rewards.unsqueeze(1).expand(-1, MAX_SEQ_LEN)
            values = torch.stack(values, dim=1)
            advantages = returns - values.detach()
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            policy_loss = -(torch.stack(log_probs, 1) * advantages).mean()
            value_loss = F.mse_loss(values, returns)
            entropy_loss = -torch.stack(log_probs, 1).mean()
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_loss

            self.opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
            self.scheduler.step()

            pbar.set_postfix({
                'Valid': f"{len(valid_idx)/B:.1%}",
                'Best': f"{self.best_sharpe:.3f}",
                'Loss': f"{loss.item():.3f}",
                'Patience': f"{self.patience_counter}/{self.max_patience}"
            })

            if self.patience_counter > self.max_patience:
                print(f"\n⏹️ 早停于迭代 {iteration}")
                break

    def decode(self, tokens=None):
        if tokens is None:
            tokens = self.best_formula_tokens
        if tokens is None:
            return "无有效公式"
        stream = list(tokens)
        stream.reverse()
        def _parse():
            if not stream:
                return ""
            t = stream.pop()
            if t < len(self.features_list):
                return self.features_list[t]
            else:
                arity = self.op_arity_map[t]
                args = [_parse() for _ in range(arity)]
                return f"{self.op_func_map[t].__name__ if hasattr(self.op_func_map[t], '__name__') else 'OP'}({', '.join(args)})"
        try:
            return _parse()
        except:
            return "解析失败"

# ==================== 策略集成管理器 ====================
class EnsembleStrategyManager:
    def __init__(self, miner, engine, top_k=20):
        self.miner = miner
        self.engine = engine
        self.top_k = top_k
        self.strategies = []
        self.ensemble_model = None
        self.strategy_weights = None

    def collect_top_strategies(self, n_candidates=1000):
        print(f"📊 收集候选策略，候选数={n_candidates}...")
        pool = []
        pbar = tqdm(range(n_candidates), desc="生成策略")
        for _ in pbar:
            tokens = self.miner.generate_random_formula()
            factor = self.miner.solve_one(tokens)
            if factor is not None:
                reward = self.miner.backtest_with_risk_control(factor.unsqueeze(0), is_train=True)[0].item()
                if reward > 0:
                    pool.append({
                        'tokens': tokens,
                        'factor': factor,
                        'reward': reward
                    })
        pool.sort(key=lambda x: x['reward'], reverse=True)
        self.strategies = pool[:self.top_k]
        print(f"✅ 收集到 {len(self.strategies)} 个高质量策略，平均奖励：{np.mean([s['reward'] for s in self.strategies]):.3f}")

    def train_ensemble(self):
        print("🤝 训练LightGBM集成模型...")
        split = self.engine.split_idx
        n_strategies = len(self.strategies)
        if n_strategies < 3:
            print("⚠️  策略不足，无法进行集成")
            return

        # 构建特征矩阵（每个样本是所有策略在当天的因子值）
        X_train = []
        for s in self.strategies:
            factor = s['factor'][:split].cpu().numpy()
            X_train.append(factor)
        X_train = np.array(X_train).T  # [天数, 策略数]
        y_train = self.engine.target_oto_ret[:split].cpu().numpy()

        train_size = int(0.8 * len(X_train))
        X_tr, X_val = X_train[:train_size], X_train[train_size:]
        y_tr, y_val = y_train[:train_size], y_train[train_size:]

        lgb_train = lgb.Dataset(X_tr, y_tr)
        lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

        # 使用Optuna优化集成模型
        def objective(trial):
            params = {
                'boosting_type': 'gbdt',
                'objective': 'regression',
                'metric': 'rmse',
                'num_leaves': trial.suggest_int('num_leaves', 20, 50),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.9),
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 50),
                'verbosity': -1,
                'seed': 42
            }
            cv = TimeSeriesSplit(n_splits=3)
            scores = []
            for train_idx, val_idx in cv.split(X_tr):
                X_tr_cv, X_val_cv = X_tr[train_idx], X_tr[val_idx]
                y_tr_cv, y_val_cv = y_tr[train_idx], y_tr[val_idx]
                lgb_train_cv = lgb.Dataset(X_tr_cv, y_tr_cv)
                lgb_val_cv = lgb.Dataset(X_val_cv, y_val_cv, reference=lgb_train_cv)
                model = lgb.train(params, lgb_train_cv, valid_sets=[lgb_val_cv], num_boost_round=100,
                                  callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)])
                scores.append(model.best_score['valid_0']['rmse'])
            return np.mean(scores)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=10, show_progress_bar=True)
        best_params = study.best_params
        best_params.update({
            'boosting_type': 'gbdt',
            'objective': 'regression',
            'metric': 'rmse',
            'verbosity': -1,
            'seed': 42
        })
        print(f"   最佳集成参数: {best_params}")

        self.ensemble_model = lgb.train(
            best_params,
            lgb_train,
            num_boost_round=300,
            valid_sets=[lgb_val],
            callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
        )

        importance = self.ensemble_model.feature_importance(importance_type='gain')
        self.strategy_weights = importance / (importance.sum() + 1e-6)
        print(f"✅ 集成模型训练完成，策略权重：{self.strategy_weights[:5]}...")

    def ensemble_backtest_detailed(self):
        """返回包含每日交易明细的DataFrame，并应用风控规则"""
        if self.ensemble_model is None:
            print("⚠️  集成模型未训练")
            return None

        split = self.engine.split_idx
        dates = self.engine.dates[split:]
        open_prices = self.engine.raw_open[split:].cpu().numpy()
        close_prices = self.engine.raw_close[split:].cpu().numpy()
        high_prices = self.engine.raw_high[split:].cpu().numpy()
        low_prices = self.engine.raw_low[split:].cpu().numpy()
        target_ret = self.engine.target_oto_ret[split:].cpu().numpy()

        # 构建特征矩阵（策略因子）
        X_test = np.array([s['factor'][split:].cpu().numpy() for s in self.strategies]).T
        predictions = self.ensemble_model.predict(X_test)

        # 基础信号
        signals = np.tanh(predictions * 0.5)

        # 计算历史波动率用于仓位调整
        returns_hist = (close_prices[1:] - close_prices[:-1]) / (close_prices[:-1] + 1e-6)
        vol_vals = pd.Series(returns_hist).rolling(20).std().fillna(0).values
        hist_vol = np.zeros_like(close_prices)
        hist_vol[20:] = vol_vals[19:] * np.sqrt(252)

        # 仓位调整
        vol_scale = VOLATILITY_TARGET / (hist_vol + 1e-6)
        vol_scale = np.clip(vol_scale, 0.2, 2.0)
        positions = signals * vol_scale

        # 止损逻辑
        daily_pnl = positions * target_ret
        stop_loss_trigger = (daily_pnl < -STOP_LOSS).astype(float)
        positions = positions * (1 - stop_loss_trigger)

        # 持仓天数限制
        pos_sign = np.sign(positions)
        pos_days = np.zeros_like(pos_sign)
        for t in range(1, len(pos_sign)):
            if pos_sign[t] == pos_sign[t-1] and pos_sign[t] != 0:
                pos_days[t] = pos_days[t-1] + 1
            else:
                pos_days[t] = 0
        force_close = (pos_days > MAX_POSITION_DAYS).astype(float)
        positions = positions * (1 - force_close)

        turnover = np.abs(positions - np.roll(positions, 1))
        turnover[0] = 0

        daily_ret = positions * target_ret - turnover * (COST_RATE + SLIPPAGE)
        equity = (1 + daily_ret).cumprod()

        # 基准净值
        bench_ret = np.diff(close_prices, prepend=close_prices[0]) / close_prices
        bench_equity = (1 + bench_ret).cumprod()

        df = pd.DataFrame({
            'date': dates,
            'open': open_prices,
            'close': close_prices,
            'position': positions,
            'signal': signals,
            'turnover': turnover,
            'daily_return': daily_ret,
            'target_return': target_ret,
            'prediction': predictions,
            'equity': equity,
            'bench_equity': bench_equity
        })
        df['trade'] = (df['position'] != df['position'].shift(1)).fillna(False)
        df['trade_price'] = df['open']
        df.loc[~df['trade'], 'trade_price'] = np.nan

        return df

    def ensemble_backtest(self):
        """集成策略回测（测试集），返回绩效字典"""
        df = self.ensemble_backtest_detailed()
        if df is None:
            return None
        daily_ret = df['daily_return'].values
        equity = df['equity'].values
        bench_equity = df['bench_equity'].values

        total_ret = equity[-1] - 1 if len(equity) > 0 else 0
        n_years = len(daily_ret) / 252
        ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
        vol = np.std(daily_ret) * np.sqrt(252)
        sharpe = (ann_ret - 0.02) / (vol + 1e-6)

        running_max = np.maximum.accumulate(equity)
        drawdown = (equity - running_max) / running_max
        max_dd = np.min(drawdown)
        calmar = ann_ret / (abs(max_dd) + 1e-6)
        win_rate = np.mean(daily_ret > 0)

        return {
            'equity': equity,
            'bench_equity': bench_equity,
            'daily_ret': daily_ret,
            'ann_ret': ann_ret,
            'vol': vol,
            'sharpe': sharpe,
            'max_dd': max_dd,
            'calmar': calmar,
            'win_rate': win_rate,
            'positions': df['position'].values,
            'df': df  # 保留详细数据用于绘图
        }

# ==================== 混合系统主控类 ====================
class AlphaGPT_LightGBM_Hybrid:
    def __init__(self):
        self.engine = None
        self.miner = None
        self.ensemble_mgr = None
        self.vocab_size = None
        self.op_func_map = None
        self.op_arity_map = None

    def _build_vocab(self, features):
        """根据特征列表动态构建词汇表"""
        vocab = features + [cfg[0] for cfg in OPS_CONFIG]
        vocab_size = len(vocab)
        op_func_map = {i + len(features): cfg[1] for i, cfg in enumerate(OPS_CONFIG)}
        op_arity_map = {i + len(features): cfg[2] for i, cfg in enumerate(OPS_CONFIG)}
        return vocab, vocab_size, op_func_map, op_arity_map

    def run(self):
        print("=" * 80)
        print("🤖 AlphaGPT + LightGBM 混合量化策略生成系统（增强版）")
        print("=" * 80)

        # ---------- 阶段1：数据加载与LightGBM特征增强 ----------
        print("\n[1/5] 数据准备与特征工程")
        self.engine = EnhancedDataEngine().load(use_lgb=True)

        # 动态构建词汇表
        features = self.engine.__class__.FEATURES
        _, vocab_size, op_func_map, op_arity_map = self._build_vocab(features)
        self.vocab_size = vocab_size
        self.op_func_map = op_func_map
        self.op_arity_map = op_arity_map

        # ---------- 阶段2：训练奖励预测器 ----------
        print("\n[2/5] 训练AlphaGPT与奖励预测器")
        self.miner = DeepQuantMiner(
            self.engine,
            vocab_size,
            op_func_map,
            op_arity_map,
            features
        )
        self.miner.train_reward_predictor(n_samples=2000)
        self.miner.train(use_reward_predictor=True)

        # ---------- 阶段3：收集最佳策略池 ----------
        print("\n[3/5] 策略池构建")
        self.ensemble_mgr = EnsembleStrategyManager(self.miner, self.engine, top_k=20)
        self.ensemble_mgr.collect_top_strategies(n_candidates=800)

        # ---------- 阶段4：训练集成模型 ----------
        print("\n[4/5] 策略集成训练")
        self.ensemble_mgr.train_ensemble()

        # ---------- 阶段5：回测与可视化 ----------
        print("\n[5/5] 回测与可视化报告")
        self._final_report()

        print("\n✅ 全部流程执行完毕！")

    def _final_report(self):
        """生成完整的回测报告与可视化图表"""
        # 1. 单策略表现（最佳策略）
        single_factor = self.miner.solve_one(self.miner.best_formula_tokens)
        single_results = None
        if single_factor is not None:
            daily_ret_single = self._backtest_single(single_factor)
            if daily_ret_single is not None:
                single_results = self._compute_metrics(daily_ret_single)
        # 2. 集成策略详细回测
        ensemble_results = self.ensemble_mgr.ensemble_backtest()
        if ensemble_results is not None:
            ensemble_df = ensemble_results['df']
            ensemble_df.to_csv('trade_log.csv', index=False)
            print("📄 交易日志已保存至 trade_log.csv")
        else:
            ensemble_df = None

        # 3. 交互式绘图（保存HTML和PNG）
        self._plot_interactive(ensemble_df=ensemble_df)

        # 4. 打印绩效表格
        self._print_performance_table(single_results, ensemble_results)

    def _backtest_single(self, factor):
        """单个策略回测（测试集）"""
        split = self.engine.split_idx
        test_factors = factor[split:].cpu().numpy()
        test_ret = self.engine.target_oto_ret[split:].cpu().numpy()
        close_prices = self.engine.raw_close[split:].cpu().numpy()

        # 历史波动率
        returns_hist = (close_prices[1:] - close_prices[:-1]) / (close_prices[:-1] + 1e-6)
        vol_vals = pd.Series(returns_hist).rolling(20).std().fillna(0).values
        hist_vol = np.zeros_like(close_prices)
        hist_vol[20:] = vol_vals[19:] * np.sqrt(252)

        smoothed = pd.Series(test_factors).rolling(3).mean().fillna(0).values
        signals = np.tanh(smoothed * 0.5)
        vol_scale = VOLATILITY_TARGET / (hist_vol + 1e-6)
        vol_scale = np.clip(vol_scale, 0.2, 2.0)
        positions = signals * vol_scale

        # 止损
        daily_pnl = positions * test_ret
        stop_loss_trigger = (daily_pnl < -STOP_LOSS).astype(float)
        positions = positions * (1 - stop_loss_trigger)

        # 持仓天数限制
        pos_sign = np.sign(positions)
        pos_days = np.zeros_like(pos_sign)
        for t in range(1, len(pos_sign)):
            if pos_sign[t] == pos_sign[t-1] and pos_sign[t] != 0:
                pos_days[t] = pos_days[t-1] + 1
            else:
                pos_days[t] = 0
        force_close = (pos_days > MAX_POSITION_DAYS).astype(float)
        positions = positions * (1 - force_close)

        turnover = np.abs(positions - np.roll(positions, 1))
        turnover[0] = 0
        daily_ret = positions * test_ret - turnover * (COST_RATE + SLIPPAGE)
        return daily_ret

    def _compute_metrics(self, daily_ret):
        """计算绩效指标，daily_ret为numpy数组"""
        equity = (1 + daily_ret).cumprod()
        total_ret = equity[-1] - 1 if len(equity) > 0 else 0
        n_years = len(daily_ret) / 252
        ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
        vol = np.std(daily_ret) * np.sqrt(252)
        sharpe = (ann_ret - 0.02) / (vol + 1e-6)
        running_max = np.maximum.accumulate(equity)
        drawdown = (equity - running_max) / running_max
        max_dd = np.min(drawdown)
        calmar = ann_ret / (abs(max_dd) + 1e-6)
        win_rate = np.mean(daily_ret > 0)
        return {
            'equity': equity,
            'ann_ret': ann_ret,
            'vol': vol,
            'sharpe': sharpe,
            'max_dd': max_dd,
            'calmar': calmar,
            'win_rate': win_rate
        }

    def _plot_interactive(self, ensemble_df=None):
        """使用Plotly生成交互式图表，并保存为HTML和PNG"""
        if ensemble_df is None:
            print("没有回测数据，无法绘图")
            return

        # 计算回撤
        running_max = ensemble_df['equity'].cummax()
        ensemble_df['drawdown'] = (ensemble_df['equity'] - running_max) / running_max

        fig = make_subplots(
            rows=4, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.4, 0.2, 0.2, 0.2],
            subplot_titles=("净值曲线", "回撤曲线", "仓位与交易点", "月度收益热图")
        )

        # 净值曲线
        fig.add_trace(go.Scatter(x=ensemble_df['date'], y=ensemble_df['equity'],
                                mode='lines', name='集成策略', line=dict(color='darkblue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=ensemble_df['date'], y=ensemble_df['bench_equity'],
                                mode='lines', name='基准', line=dict(color='gray', dash='dash')), row=1, col=1)

        # 回撤曲线
        fig.add_trace(go.Scatter(x=ensemble_df['date'], y=ensemble_df['drawdown']*100,
                                fill='tozeroy', mode='lines', name='回撤', line=dict(color='crimson')), row=2, col=1)

        # 仓位与交易点
        fig.add_trace(go.Scatter(x=ensemble_df['date'], y=ensemble_df['position']*100,
                                mode='lines', name='仓位(%)', line=dict(color='seagreen')), row=3, col=1)

        # 交易点标记
        trade_df = ensemble_df[ensemble_df['trade']]
        if not trade_df.empty:
            fig.add_trace(go.Scatter(x=trade_df['date'], y=[100]*len(trade_df),  # 用顶部位置显示
                                    mode='markers', name='交易点',
                                    marker=dict(color='red', size=8, symbol='triangle-down'),
                                    hovertemplate='交易日期: %{x}<br>交易价格: %{customdata:.2f}<extra></extra>',
                                    customdata=trade_df['trade_price']), row=3, col=1)

        # 月度收益热图（将收益率按月份聚合）
        df_plot = ensemble_df.copy()
        df_plot['year'] = pd.DatetimeIndex(df_plot['date']).year
        df_plot['month'] = pd.DatetimeIndex(df_plot['date']).month
        monthly_ret = df_plot.groupby(['year', 'month'])['daily_return'].apply(lambda x: (1+x).prod() - 1).reset_index()
        pivot = monthly_ret.pivot(index='year', columns='month', values='daily_return').fillna(0)
        months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
        pivot.columns = [months[i-1] for i in pivot.columns]

        heatmap = go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            colorscale='RdYlGn',
            zmid=0,
            text=np.round(pivot.values*100, 1),
            texttemplate='%{text}%',
            textfont={"size":10},
            colorbar_title="月收益"
        )
        fig.add_trace(heatmap, row=4, col=1)

        fig.update_layout(height=1000, title_text="集成策略回测详情", hovermode='x unified')
        # 保存为HTML
        pio.write_html(fig, file='ensemble_backtest.html', auto_open=False)
        print("📁 交互式图表已保存为 ensemble_backtest.html")
        # 也可保存为静态图片（需要安装 kaleido）
        try:
            pio.write_image(fig, 'ensemble_backtest.png', width=1200, height=800)
            print("📸 静态图片已保存为 ensemble_backtest.png")
        except:
            print("⚠️  kaleido未安装，无法保存静态图片")
        fig.show()

    def _print_performance_table(self, single, ensemble):
        """打印绩效表格"""
        print("\n" + "="*80)
        print("📊 最终绩效汇总（测试集）")
        print("="*80)
        print(f"{'指标':<20} {'最佳单策略':<20} {'集成策略':<20} {'基准':<20}")
        print("-"*80)

        # 基准
        split = self.engine.split_idx
        close_prices = self.engine.raw_close[split:].cpu().numpy()
        bench_ret = np.zeros_like(close_prices)
        bench_ret[1:] = (close_prices[1:] - close_prices[:-1]) / close_prices[:-1]
        bench_metrics = self._compute_metrics(bench_ret)

        ann_ret_s = single['ann_ret'] if single else 0
        ann_ret_e = ensemble['ann_ret'] if ensemble else 0
        ann_ret_b = bench_metrics['ann_ret']

        vol_s = single['vol'] if single else 0
        vol_e = ensemble['vol'] if ensemble else 0
        vol_b = bench_metrics['vol']

        sharpe_s = single['sharpe'] if single else 0
        sharpe_e = ensemble['sharpe'] if ensemble else 0
        sharpe_b = bench_metrics['sharpe']

        dd_s = single['max_dd'] if single else 0
        dd_e = ensemble['max_dd'] if ensemble else 0
        dd_b = bench_metrics['max_dd']

        calmar_s = single['calmar'] if single else 0
        calmar_e = ensemble['calmar'] if ensemble else 0
        calmar_b = bench_metrics['calmar']

        win_s = single['win_rate'] if single else 0
        win_e = ensemble['win_rate'] if ensemble else 0
        win_b = bench_metrics['win_rate']

        print(f"{'年化收益率':<20} {ann_ret_s:>19.2%} {ann_ret_e:>19.2%} {ann_ret_b:>19.2%}")
        print(f"{'年化波动率':<20} {vol_s:>19.2%} {vol_e:>19.2%} {vol_b:>19.2%}")
        print(f"{'夏普比率':<20} {sharpe_s:>19.2f} {sharpe_e:>19.2f} {sharpe_b:>19.2f}")
        print(f"{'最大回撤':<20} {dd_s:>19.2%} {dd_e:>19.2%} {dd_b:>19.2%}")
        print(f"{'卡玛比率':<20} {calmar_s:>19.2f} {calmar_e:>19.2f} {calmar_b:>19.2f}")
        print(f"{'胜率':<20} {win_s:>19.2%} {win_e:>19.2%} {win_b:>19.2%}")
        print("="*80)

        # 最佳公式
        print("\n🧬 最佳策略公式：")
        print(self.miner.decode())

# ==================== 主程序入口 ====================
if __name__ == "__main__":
    hybrid = AlphaGPT_LightGBM_Hybrid()
    hybrid.run()

[I 2026-03-04 11:36:45,313] A new study created in memory with name: no-name-afd368ae-347a-4729-a109-cd22826ceea5


🤖 AlphaGPT + LightGBM 混合量化策略生成系统（增强版）

[1/5] 数据准备与特征工程
✅ 基础数据加载完成，样本数：2688，训练集：0-2150，测试集：2150-2688
🔍 训练LightGBM进行特征选择与组合...


  0%|          | 0/20 [00:00<?, ?it/s]

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.0142452
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.014699
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.0115741
[I 2026-03-04 11:36:45,334] Trial 0 finished with value: 0.013506121622820546 and parameters: {'num_leaves': 74, 'learning_rate': 0.01186980842390953, 'feature_fraction': 0.971517168545895, 'bagging_fraction': 0.6969671932676311, 'bagging_freq': 5, 'min_data_in_leaf': 45}. Best is trial 0 with value: 0.013506121622820546.
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[10]	valid_0's rmse: 0.0141152
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 0.0146506
Training until validation scores don't improve

生成样本: 100%|██████████| 2000/2000 [02:06<00:00, 15.86it/s]


...

生成策略: 100%|██████████| 800/800 [00:50<00:00, 15.70it/s]
[I 2026-03-04 11:41:01,572] A new study created in memory with name: no-name-0d5ccb57-964a-4c2f-ae45-cdced0742ab1


✅ 收集到 20 个高质量策略，平均奖励：10.000

[4/5] 策略集成训练
🤝 训练LightGBM集成模型...


  0%|          | 0/10 [00:00<?, ?it/s]

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[73]	valid_0's rmse: 0.00435409
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.00657281
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.00348602
[I 2026-03-04 11:41:01,766] Trial 0 finished with value: 0.004804309432014429 and parameters: {'num_leaves': 36, 'learning_rate': 0.02126846602181479, 'feature_fraction': 0.7340000303999195, 'bagging_fraction': 0.6490913926010934, 'bagging_freq': 4, 'min_data_in_leaf': 28}. Best is trial 0 with value: 0.004804309432014429.
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.0053137
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0


📊 最终绩效汇总（测试集）
指标                   最佳单策略                集成策略                 基准                  
--------------------------------------------------------------------------------
年化收益率                             -1.20%               0.07%              20.72%
年化波动率                              7.39%               0.10%              24.13%
夏普比率                               -0.43              -19.38                0.78
最大回撤                             -10.55%              -0.09%             -21.43%
卡玛比率                               -0.11                0.77                0.97
胜率                                38.85%              45.35%              50.56%

🧬 最佳策略公式：
LGB_FEATURE

✅ 全部流程执行完毕！
